In [6]:
import os
from pathlib import Path
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam

In [9]:
# Definir rutas relativas a partir del directorio base
BASE_DIR = Path("..") 
DATA_RAW_DIR = BASE_DIR / "data" / "raw"
DATASET_DIR = DATA_RAW_DIR / "waste_classification"
MODELS_DIR = BASE_DIR / "models"

TRAIN_DIR = DATASET_DIR / "train"
TEST_DIR = DATASET_DIR / "test"

# Crear carpeta de modelos si no existe
MODELS_DIR.mkdir(parents=True, exist_ok=True)

# Verificación estricta de la existencia de los datos
if not TRAIN_DIR.exists() or not TEST_DIR.exists():
    print(f"⚠️ ¡Atención! No se encontró la estructura esperada en {DATASET_DIR}")
    print("Asegúrate de colocar tus carpetas 'train' y 'test' manualmente ahí.")
else:
    print(f"✅ Directorios de datos detectados correctamente en:\n - {TRAIN_DIR}\n - {TEST_DIR}")

✅ Directorios de datos detectados correctamente en:
 - ..\data\raw\waste_classification\train
 - ..\data\raw\waste_classification\test


In [10]:
IMG_SIZE = (224, 224)
BATCH_SIZE = 16

# 1. Generador de Entrenamiento (CON Data Augmentation)
train_datagen = ImageDataGenerator(
    rescale=1./255,         
    rotation_range=30,      
    width_shift_range=0.2,  
    height_shift_range=0.2, 
    zoom_range=0.3,         
    horizontal_flip=True    
)

# 2. Generador de Prueba/Validación (SIN alteraciones, solo normalizado matemático)
test_datagen = ImageDataGenerator(rescale=1./255)

print("Cargando set de ENTRENAMIENTO...")
train_gen = train_datagen.flow_from_directory(
    str(TRAIN_DIR),
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=True # Fundamental mezclar para entrenar
)

print("Cargando set de PRUEBA/VALIDACIÓN...")
val_gen = test_datagen.flow_from_directory(
    str(TEST_DIR),
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False # No se mezcla para poder evaluar ordenadamente
)

NUM_CLASSES = train_gen.num_classes
print(f"\nTotal de clases detectadas: {NUM_CLASSES}")
print(f"Diccionario de clases: {train_gen.class_indices}")

Cargando set de ENTRENAMIENTO...
Found 22787 images belonging to 5 classes.
Cargando set de PRUEBA/VALIDACIÓN...
Found 5700 images belonging to 5 classes.

Total de clases detectadas: 5
Diccionario de clases: {'crushed_metal': 0, 'crushed_plastic': 1, 'metal': 2, 'no_reciclable': 3, 'plastic': 4}


In [11]:
print("Cargando modelo base MobileNetV2...")
base_model = MobileNetV2(
    weights='imagenet',       
    include_top=False,        
    input_shape=(224, 224, 3) 
)

# Congelamos el extractor de características
base_model.trainable = False

# Construcción de la nueva cabecera
x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dense(128, activation='relu')(x) 
x = Dropout(0.3)(x) 

# Salida dinámica para nuestras 5 categorías
output = Dense(NUM_CLASSES, activation='softmax')(x)

model = Model(inputs=base_model.input, outputs=output)
print("✅ Arquitectura finalizada adaptada a 5 clases.")

Cargando modelo base MobileNetV2...
✅ Arquitectura finalizada adaptada a 5 clases.


In [12]:
import tensorflow as tf
from tensorflow.keras.applications import MobileNetV2

# Configurar el modelo base pre-entrenado (MobileNetV2)
print("Descargando/Cargando modelo MobileNetV2 (esto puede tardar un poco la primera vez)...")

# Instanciamos el modelo MobileNetV2
base_model = MobileNetV2(
    weights='imagenet',       # Usamos el conocimiento previo de ImageNet
    include_top=False,        # ¡Crucial! Quitamos la capa de clasificación final (la "cabeza")
    input_shape=(224, 224, 3) # Definimos que entrarán imágenes de 224x224 con 3 canales (RGB)
)

# --- CONGELAMIENTO DE CAPAS ---
# "Congelamos" el modelo base. Esto significa que sus pesos (números internos)
# NO cambiarán durante la primera fase del entrenamiento.
# Esto nos permite usarlo solo como un "extractor de características" ultra avanzado.
base_model.trainable = False

print("✅ Modelo base cargado y congelado correctamente.")
print(f"   Estructura de entrada: {base_model.input_shape}")

Descargando/Cargando modelo MobileNetV2 (esto puede tardar un poco la primera vez)...
✅ Modelo base cargado y congelado correctamente.
   Estructura de entrada: (None, 224, 224, 3)


In [13]:
import tensorflow as tf
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout
from tensorflow.keras.models import Model

# Construcción de la nueva cabecera del modelo
x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dense(128, activation='relu')(x) # Capa intermedia para aprender patrones complejos
x = Dropout(0.3)(x) # Apaga el 30% de neuronas al azar para evitar memorización (overfitting)

# Capa de salida: NUM_CLASSES debe ser 30 (automático gracias al paso anterior)
output = Dense(NUM_CLASSES, activation='softmax')(x)

# Ensamblamos el modelo final
model = Model(inputs=base_model.input, outputs=output)

print("Arquitectura del modelo finalizada.")
model.summary() # Muestra un resumen para verificar que todo conecta bien

Arquitectura del modelo finalizada.


Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_1       │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Conv1 (Conv2D)      │ (None, 112, 112,  │        864 │ input_layer_1[0]… │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bn_Conv1            │ (None, 112, 112,  │        128 │ Conv1[0][0]       │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Conv1_relu (ReLU)   │ (None, 112, 112,  │          0 │ bn_Conv1[0][0]    │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_dept… │ (None, 112, 112,  │        288 │ Conv1_relu[0][0]  │
│ (DepthwiseConv2D)   │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_dept… │ (None, 112, 112,  │        128 │ expanded_conv_de… │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_dept… │ (None, 112, 112,  │          0 │ expanded_conv_de… │
│ (ReLU)              │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_proj… │ (None, 112, 112,  │        512 │ expanded_conv_de… │
│ (Conv2D)            │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_proj… │ (None, 112, 112,  │         64 │ expanded_conv_pr… │
│ (BatchNormalizatio… │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_expand      │ (None, 112, 112,  │      1,536 │ expanded_conv_pr… │
│ (Conv2D)            │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_expand_BN   │ (None, 112, 112,  │        384 │ block_1_expand[0… │
│ (BatchNormalizatio… │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_expand_relu │ (None, 112, 112,  │          0 │ block_1_expand_B… │
│ (ReLU)              │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_pad         │ (None, 113, 113,  │          0 │ block_1_expand_r… │
│ (ZeroPadding2D)     │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_depthwise   │ (None, 56, 56,    │        864 │ block_1_pad[0][0] │
│ (DepthwiseConv2D)   │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_depthwise_… │ (None, 56, 56,    │        384 │ block_1_depthwis… │
│ (BatchNormalizatio… │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_depthwise_… │ (None, 56, 56,    │          0 │ block_1_depthwis… │
│ (ReLU)              │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_project     │ (None, 56, 56,    │      2,304 │ block_1_depthwis

 Total params: 2,422,597 (9.24 MB)

 Trainable params: 164,613 (643.02 KB)

 Non-trainable params: 2,257,984 (8.61 MB)

In [14]:
import tensorflow as tf
print("Versión de TensorFlow:", tf.__version__)
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    print(f"✅ GPU Detectada: {gpus}")
else:
    print("⚠️ No se detectó GPU. El entrenamiento correrá en CPU (Lento).")

Versión de TensorFlow: 2.20.0
⚠️ No se detectó GPU. El entrenamiento correrá en CPU (Lento).


In [15]:
import tensorflow as tf
from tensorflow.keras.optimizers import Adam

# COMPILACIÓN DEL MODELO 
# Definimos cómo va a aprender la red neuronal
print("Compilando el modelo con optimizador Adam...")

model.compile(
    optimizer=Adam(learning_rate=1e-3), # 0.001: Velocidad de aprendizaje inicial estándar
    loss='categorical_crossentropy',    # Obligatorio para clasificación de múltiples clases (>2)
    metrics=['accuracy']                # Queremos monitorear qué tan preciso es (0.0 a 1.0)
)

# ENTRENAMIENTO FITTING 
print("\nIniciando entrenamiento (Fase 1 - Cabecera)...")
print("   Nota: Esto puede tardar dependiendo de tu hardware (CPU vs GPU).")

history = model.fit(
    train_gen,               # Datos de entrenamiento (con data augmentation)
    validation_data=val_gen, # Datos de validación (para examen final de cada época)
    epochs=15,               # Número de vueltas completas al dataset
    verbose=1                # Muestra una barra de progreso animada
)

print("¡Entrenamiento fase 1 completado!")

Compilando el modelo con optimizador Adam...

Iniciando entrenamiento (Fase 1 - Cabecera)...
   Nota: Esto puede tardar dependiendo de tu hardware (CPU vs GPU).
Epoch 1/15
1425/1425 ━━━━━━━━━━━━━━━━━━━━ 433s 301ms/step - accuracy: 0.9254 - loss: 0.2303 - val_accuracy: 0.9600 - val_loss: 0.1207
Epoch 2/15
1425/1425 ━━━━━━━━━━━━━━━━━━━━ 393s 276ms/step - accuracy: 0.9472 - loss: 0.1569 - val_accuracy: 0.9646 - val_loss: 0.1081
Epoch 3/15
1425/1425 ━━━━━━━━━━━━━━━━━━━━ 412s 289ms/step - accuracy: 0.9562 - loss: 0.1366 - val_accuracy: 0.9707 - val_loss: 0.0930
Epoch 4/15
1425/1425 ━━━━━━━━━━━━━━━━━━━━ 396s 278ms/step - accuracy: 0.9573 - loss: 0.1292 - val_accuracy: 0.9696 - val_loss: 0.0899
Epoch 5/15
1425/1425 ━━━━━━━━━━━━━━━━━━━━ 397s 279ms/step - accuracy: 0.9598 - loss: 0.1193 - val_accuracy: 0.9725 - val_loss: 0.0855
Epoch 6/15
1425/1425 ━━━━━━━━━━━━━━━━━━━━ 396s 278ms/step - accuracy: 0.9649 - loss: 0.1074 - val_accuracy: 0.9756 - val_loss: 0.0823
Epoch 7/15
1425/1425 ━━━━━━━━━━━━━━

In [16]:
import tensorflow as tf

print("Iniciando configuración de Fine-Tuning...")

# Descongelamos todo el modelo base primero
base_model.trainable = True

# Decidimos dónde cortar: Congelaremos el 70% inferior
fine_tune_at = int(len(base_model.layers) * 0.7)

# Congelamos las capas anteriores al punto de corte
for layer in base_model.layers[:fine_tune_at]:
    layer.trainable = False

print(f"   Total de capas en MobileNetV2: {len(base_model.layers)}")
print(f"   Capas congeladas (70%): {fine_tune_at}")
print(f"   Capas entrenables (30%): {len(base_model.layers) - fine_tune_at}")

# RE-COMPILACIÓN (Obligatorio después de descongelar)
# ¡IMPORTANTE! Usamos un Learning Rate MUY BAJO (1e-5)
# Esto es para hacer "micro-ajustes" sin romper los pesos ya aprendidos.
model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-5),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

print("\nIniciando Fase 2 de Entrenamiento (Fine-Tuning)...")
print("   Nota: Esto tomará más tiempo por época, pero mejorará la precisión.")

# Entrenamiento final
history_fine = model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=20 # Entrenamos por 20 épocas más
)

print("¡Entrenamiento completo finalizado!")

Iniciando configuración de Fine-Tuning...
   Total de capas en MobileNetV2: 154
   Capas congeladas (70%): 107
   Capas entrenables (30%): 47

Iniciando Fase 2 de Entrenamiento (Fine-Tuning)...
   Nota: Esto tomará más tiempo por época, pero mejorará la precisión.
Epoch 1/20
1425/1425 ━━━━━━━━━━━━━━━━━━━━ 519s 358ms/step - accuracy: 0.9465 - loss: 0.1793 - val_accuracy: 0.9770 - val_loss: 0.0769
Epoch 2/20
1425/1425 ━━━━━━━━━━━━━━━━━━━━ 493s 346ms/step - accuracy: 0.9614 - loss: 0.1195 - val_accuracy: 0.9793 - val_loss: 0.0786
Epoch 3/20
1425/1425 ━━━━━━━━━━━━━━━━━━━━ 480s 337ms/step - accuracy: 0.9645 - loss: 0.1071 - val_accuracy: 0.9798 - val_loss: 0.0696
Epoch 4/20
1425/1425 ━━━━━━━━━━━━━━━━━━━━ 486s 341ms/step - accuracy: 0.9703 - loss: 0.0907 - val_accuracy: 0.9798 - val_loss: 0.0701
Epoch 5/20
1425/1425 ━━━━━━━━━━━━━━━━━━━━ 465s 327ms/step - accuracy: 0.9733 - loss: 0.0856 - val_accuracy: 0.9795 - val_loss: 0.0735
Epoch 6/20
1425/1425 ━━━━━━━━━━━━━━━━━━━━ 463s 325ms/step - accur

In [17]:
import os

# GUARDADO DEL MODELO 

# Aseguramos que el directorio de modelos exista
# Si un compañero borró la carpeta 'models' o no se bajó de git, esto la crea.
if not MODELS_DIR.exists():
    print(f"La carpeta {MODELS_DIR} no existía. Creándola...")
    MODELS_DIR.mkdir(parents=True, exist_ok=True)

# Definimos la ruta completa
# Usamos la extensión .keras que es el estándar actual más seguro y ligero que .h5
model_save_path = MODELS_DIR / "modelo_residuos_rpi_02.keras"

print(f"Guardando modelo maestro en: {model_save_path} ...")

# Guardamos el archivo
model.save(model_save_path)

# Confirmación visual con ruta absoluta para evitar confusiones sobre dónde se guardó el modelo
print(f"¡Éxito! El archivo está listo.")
print(f"   Ruta absoluta: {model_save_path.resolve()}")

Guardando modelo maestro en: ..\models\modelo_residuos_rpi_02.keras ...
¡Éxito! El archivo está listo.
   Ruta absoluta: C:\Users\river\Documents\GitHub\EntrenamientoIA\models\modelo_residuos_rpi_02.keras


In [18]:
print("Iniciando conversión a TFLite...")

# Cargar el convertidor desde el modelo Keras en memoria
converter = tf.lite.TFLiteConverter.from_keras_model(model)

# Aplicar optimizaciones Cuantización
# Esto reduce el tamaño del archivo y acelera la inferencia en la Raspberry Pi
# sin sacrificar casi nada de precisión.
converter.optimizations = [tf.lite.Optimize.DEFAULT]

# Realizar la conversión
tflite_model = converter.convert()

# Guardar el archivo .tflite
tflite_save_path = MODELS_DIR / "modelo_residuos_rpi_02.tflite"

with open(tflite_save_path, "wb") as f:
    f.write(tflite_model)

print(f"Modelo TFLite guardado en: {tflite_save_path}")

# --- COMPARACIÓN DE TAMAÑOS ---
# Esto es meramente para mostrar en el reporte final
keras_size = os.path.getsize(model_save_path) / (1024 * 1024)
tflite_size = os.path.getsize(tflite_save_path) / (1024 * 1024)

print(f"\nReporte de Optimización:")
print(f"   Tamaño Original (.keras): {keras_size:.2f} MB")
print(f"   Tamaño Optimizado (.tflite): {tflite_size:.2f} MB")
print(f"   Reducción de tamaño: {((keras_size - tflite_size) / keras_size) * 100:.2f}% menos")

Iniciando conversión a TFLite...
INFO:tensorflow:Assets written to: C:\Users\river\AppData\Local\Temp\tmpfhx5l63i\assets


INFO:tensorflow:Assets written to: C:\Users\river\AppData\Local\Temp\tmpfhx5l63i\assets


Saved artifact at 'C:\Users\river\AppData\Local\Temp\tmpfhx5l63i'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 224, 224, 3), dtype=tf.float32, name='keras_tensor_158')
Output Type:
  TensorSpec(shape=(None, 5), dtype=tf.float32, name=None)
Captures:
  2325486494992: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2325486496912: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2325486496720: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2325486494800: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2325486497488: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2325486494608: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2325486497104: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2325486497296: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2325486491536: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2325486498448: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2